# Nhóm 4 — Callbacks và TensorBoard trên Kaggle
Notebook là adapter nền tảng, toàn bộ pipeline dùng chung `src/` và `scripts/run_group4_utilities.py`; seed=42, CUDA được chọn tự động nếu runtime cấp GPU.

In [ ]:
from pathlib import Path
import os,sys,shutil,torch
PROJECT_ROOT=Path('/kaggle/working/BTL03'); INPUT_ROOT=Path('/kaggle/input'); WORK_ROOT=Path('/kaggle/working')
if not (PROJECT_ROOT/'scripts/run_group4_utilities.py').is_file(): raise FileNotFoundError('Checkout BTL03 vào /kaggle/working/BTL03.')
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT))
from src.utils import seed_everything
seed_everything(42)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Kaggle input mounts:',[p.name for p in INPUT_ROOT.iterdir()] if INPUT_ROOT.exists() else [])

## Ánh xạ dataset đã attach
Tên file dự kiến: WIKI `train_x.npy`, `train_y.npy`, `validation_x.npy`, `validation_y.npy`; FER `fer2013_training_onehot.csv`, `fer2013_publictest_onehot.csv`. Dataset nguồn chỉ đọc từ `/kaggle/input`, bản copy vào working/data bị Git ignore.

In [ ]:
expected={'data/raw/wiki_gender/train_x.npy':'train_x.npy','data/raw/wiki_gender/train_y.npy':'train_y.npy','data/raw/wiki_gender/validation_x.npy':'validation_x.npy','data/raw/wiki_gender/validation_y.npy':'validation_y.npy','data/raw/fer2013/fer2013_training_onehot.csv':'fer2013_training_onehot.csv','data/raw/fer2013/fer2013_publictest_onehot.csv':'fer2013_publictest_onehot.csv'}
for relative, basename in expected.items():
    matches=list(INPUT_ROOT.rglob(basename)) if INPUT_ROOT.exists() else []
    if matches:
        target=PROJECT_ROOT/relative; target.parent.mkdir(parents=True,exist_ok=True)
        if not target.exists(): shutil.copy2(matches[0],target)
!python scripts/download_data.py check

In [ ]:
!python scripts/run_group4_utilities.py --task all

In [ ]:
from IPython.display import display,Image
for task in ['gender','emotion']:
    path=PROJECT_ROOT/f'outputs/figures/t0209_{task}_utilities_curves.png'
    if path.is_file(): display(Image(filename=str(path)))

## Đóng gói artifacts
ZIP chứa checkpoint, CSV, TensorBoard events và figures; không bao gồm dữ liệu nguồn.

In [ ]:
artifact=WORK_ROOT/'group4_utilities_artifacts'; artifact.mkdir(exist_ok=True)
for relative in ['outputs/checkpoints/04_training_utilities','experiments/csv_logs','experiments/tensorboard/Group4_Utilities','outputs/figures/t0209_gender_utilities_curves.png','outputs/figures/t0209_emotion_utilities_curves.png','outputs/figures/t0209_gender_utilities_metrics.json','outputs/figures/t0209_emotion_utilities_metrics.json']:
    source=PROJECT_ROOT/relative
    if source.exists():
        target=artifact/relative.replace('/','_')
        if source.is_dir(): shutil.copytree(source,target,dirs_exist_ok=True)
        else: shutil.copy2(source,target)
zip_path=shutil.make_archive(str(WORK_ROOT/'group4_utilities_artifacts'),'zip',artifact)
print('Artifacts:',zip_path)